In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

import matplotlib.pyplot as plt
from pendulibrary.continuation import find_bifurcation, adaptive_cont
from pendulibrary.targeter import dc_underconstrained
from pendulibrary.common_targetters import single_fixed
from pendulibrary.plotters import plot_nu_functions, compare_fams
from pendulibrary.integrate import integrate_state
from tqdm.auto import tqdm

In [ ]:
fname = "DDsp"

data = np.load(f"../database/{fname}.npz")
x0s_in = data["x0s"]
periods_in = data["periods"]
tangents_in = data["tangents"]
eigs_in = data["eigs"]
Lr, Mr = data["params"]

targ = single_fixed(0, x0s_in[0][0], 3, Lr, Mr, 1e-14)
func = targ.g_dg_stm

In [ ]:
%matplotlib ipympl
fig = plot_nu_functions(eigs_in, 8)
plt.show()

In [ ]:
im1 = 20
im0 = 21

Xm0 = targ.get_X(x0s_in[im0], periods_in[im0])
Xm1 = targ.get_X(x0s_in[im1], periods_in[im1])

g, dg, stm = func(Xm1)
svd = np.linalg.svd(dg)
tangent = svd.Vh[-1]
sprev = np.linalg.norm(Xm0 - Xm1)
if np.dot(tangent, Xm0 - Xm1) / sprev < 0:
    tangent *= -1
sprev = np.linalg.norm(Xm0 - Xm1)
print(f"Last dist is {sprev:.3e}")

In [ ]:
X0, tan = find_bifurcation(
    Xm1,
    func,
    tangent,
    sprev / 5,
    targ_tol=1e-13,
    bisect_tol=1e-11,
    period_mult=3,
    debug=True,
    scale=5,
    seek_local_opt=False,
)

In [ ]:
X1 = X0.copy()
X1[-1] *= 3
g, dg, stm = func(X1)
tan = np.linalg.svd(dg).Vh[-2]

In [ ]:
cont_kwargs = dict(
    s0=1e-7, s_min=1e-7, S=25.0, tol=5e-10, max_iter=25, target_iter=7, rate=1.02
)
Xs, eig_vals, (DFs, tangents, stms) = adaptive_cont(
    X1, func, -tan, **cont_kwargs, exact_tangent=True
)

In [ ]:
import os

plt.close("all")
fig = compare_fams(
    Xs, [f.removesuffix(".npz") for f in os.listdir("../database/") if "DDsp" if f]
)
plt.show()

In [ ]:
x0s_new = np.array([targ.get_x0(X) for X in Xs])
periods_new = np.array([targ.get_period(X) for X in Xs])
np.savez(
    "../database/DDsp-P3b",
    x0s=x0s_new,
    periods=periods_new,
    eigs=eig_vals,
    tangents=tangents,
    params=np.array([Lr, Mr]),
)

In [ ]:
plt.close("all")
ts, xs, fs = integrate_state(x0s_new[-1], periods_new[-1], Lr, Mr, 1e-14)
y = -np.cos(xs[0]) - Lr * np.cos(xs[1])
x = np.sin(xs[0]) + Lr * np.sin(xs[1])

# plt.plot(xs.T)
plt.plot(x, y)
plt.show()

In [ ]:
from pendulibrary.plotters import make_gif

In [ ]:
make_gif(xs,ts,fs, Lr, "test.gif")